# Affine Kazhdan-Lusztig 当前工作流演示

这个 notebook 演示 `pyw` 当前已经实现、并有基础测试覆盖的 affine Kazhdan-Lusztig 工作流。

当前覆盖的能力包括：

- `KazhdanLusztigPolynomials.P(x, y)`：普通 KL 多项式
- `KazhdanLusztigPolynomials.Q(x, y)`：**Weyl 群元素级** inverse KL
- `KazhdanLusztigPolynomials.Q_tilde(x_coset, y_coset)`：**陪集级 / parabolic** inverse KL 入口
- `AffineKLContext.build(...)`：从 `(algebra, lambda_hat, order)` 构造 bounded affine workflow 上下文
- `KazhdanLusztigCharacter.character(...)`：第一版 **grade-only** character 组装

## 当前范围说明

这个 notebook **只演示当前已经完成的接口**。它不会声称已经实现一般情形下完整的 $\widehat W_\Lambda$ simple-root 恢复，也不会把当前的 grade-only character 结果包装成更丰富的按权展开特征标。

另外要特别注意：本 notebook 的 affine 部分依赖 **bounded workflow**。这里的 `order` 不只是输出到多少阶，还会影响 affine 候选集本身。


In [3]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from sage.all import WeylGroup

from pyw.core import (
    AffineLieAlgebra,
    AffineWeight,
    AffineKLContext,
    KazhdanLusztigPolynomials,
    KazhdanLusztigCharacter,
    VermaCharacter,
)

print('Imports ok')

Imports ok


## 1. 准备一个最小 affine 例子

这里使用 $A_2^{(1)}$，并取 affine fundamental weight $\widehat\Lambda_1$ 作为输入权。


In [4]:
alg = AffineLieAlgebra(['A', 2, 1])
lambda_hat = AffineWeight.affine_fundamental_weight(alg, 1)
rho_hat = alg.affine_rho()

print('algebra          =', alg)
print('lambda_hat       =', lambda_hat)
print('lambda_hat+rho   =', lambda_hat + rho_hat)
print('dynkin labels    =', lambda_hat.dynkin_labels())

algebra          = AffineLieAlgebra(['A', 2, 1])
lambda_hat       = (Lambda[1]; 1; 0)
lambda_hat+rho   = (2*Lambda[1] + Lambda[2]; 4; 0)
dynkin labels    = {0: 0, 1: 1, 2: 0}


## 2. 元素级 `Q` 与陪集级 `Qtilde` 的语义

这一点很重要：

- `Q(x, y)` 的自变量是 **Weyl 群元素**
- `Q_tilde([x], [y])` 的自变量是 **陪集对象**

当前公开 `Q_tilde(...)` 的语义仍然是陪集级对象；而本 notebook 后面在 affine 部分直接演示的，是建立在 bounded candidates 之上的辅助流程 `affine_bounded_parabolic_Q_tilde(...)`。

先用有限型 $A_2$ 演示元素级 `Q`。


In [5]:
W = WeylGroup(['A', 2])
kl_finite = KazhdanLusztigPolynomials(W)

e = W.one()
w0 = W.long_element()

print('P(e, w0)    =', kl_finite.P(e, w0, at_one=True))
print('Q(e, w0)    =', kl_finite.Q(e, w0, at_one=True))
print('Q(e, e)     =', kl_finite.Q(e, e, at_one=True))

P(e, w0)    = 1
Q(e, w0)    = 1
Q(e, e)     = 1


AttributeError: 'WeylGroup_gens_with_category' object has no attribute 'elements'

## 3. 用 `AffineKLContext` 构造有界 affine 工作流

`AffineKLContext.build(...)` 会把当前已经实现的 bounded affine workflow 的主要中间状态显式化。

当前它会给出：

- `translation_bounds`
- bounded affine candidates
- dominant `Lambda_hat`
- `w_to_Lambda` 与 `w_to_lambda`
- bounded stabilizer
- quotient representatives


In [ ]:
kl_affine = KazhdanLusztigPolynomials(WeylGroup(['A', 2, 1]))
context = AffineKLContext.build(alg, lambda_hat, order=1, kl=kl_affine)

print('translation_bounds =', context.translation_bounds)
print('Lambda_hat         =', context.Lambda_hat)
print('w_to_Lambda        =', context.w_to_Lambda.affine_word())
print('w_to_lambda        =', context.w_to_lambda.affine_word())
print('ambient size       =', len(context.ambient_candidates))
print('stabilizer size    =', len(context.stabilizer_candidates))
print('quotient reps      =', len(context.quotient_representatives))

In [ ]:
print('first few quotient representatives:')
for w in context.quotient_representatives[:8]:
    print('  affine_word =', w.affine_word(), '  image =', context.quotient_weight(w))

## 4. bounded affine coset-level `Qtilde`

当前实现里，bounded affine 的 coset-level 系数通过

- bounded candidates
- bounded stabilizer
- bounded affine `Q`

来组装。下面先看几个最简单的值。


In [ ]:
lower = context.w_to_lambda
reps = context.weyl_to_be_summed()

print('number of reps to be summed =', len(reps))
print('lower affine word            =', lower.affine_word())

for rep in reps[:5]:
    coeff = kl_affine.affine_bounded_parabolic_Q_tilde(
        kl_affine.weyl_group.from_reduced_word(lower.affine_word()),
        kl_affine.weyl_group.from_reduced_word(rep.affine_word()),
        candidates=context.ambient_candidates_sage,
        stabilizer_candidates=[
            kl_affine.weyl_group.from_reduced_word(w.affine_word())
            for w in context.stabilizer_candidates
        ],
        at_one=True,
    )
    print('rep =', rep.affine_word(), '  Qtilde =', coeff)

## 5. Verma character

当前 `VermaCharacter` 仍然是一个独立可用的接口。它给出的是 **grade-only** 形式特征标。


In [ ]:
verma = VermaCharacter(alg, context.Lambda_hat)
verma_ch = verma.character(max_grade=4)

print('Verma character coefficients up to grade 4:')
for grade, coeff in verma_ch:
    print(f'  q^{grade}: {coeff}')

## 6. 第一版 `KazhdanLusztigCharacter`

当前版本会：

1. 构造 `AffineKLContext`
2. 在 bounded affine quotient reps 上求 `Qtilde(1)`
3. 用对应权的 `VermaCharacter` 做求和
4. 返回一个 **grade-only** `FormalCharacter`

这里的 `order` 当前同时承担两件事：

- bounded affine 搜索窗口
- 最终 grade 截断上界

这是一版可运行实现，但还不是最终的一般按权 character 公式。


In [ ]:
kl_character = KazhdanLusztigCharacter(alg)
terms = kl_character.numerator_terms(lambda_hat, order=1)

print('number of numerator terms =', len(terms))
print('first few terms:')
for term in terms[:8]:
    print('  rep =', term.representative.affine_word(), ' coeff =', term.coefficient, ' weight =', term.weight)

In [ ]:
ch = kl_character.character(lambda_hat, order=3)

print('Kazhdan-Lusztig grade-only character up to grade 3:')
for grade, coeff in ch:
    print(f'  q^{grade}: {coeff}')

## 7. 当前实现的边界

使用当前接口时，需要记住下面这些边界：

- 当前 `AffineKLContext` 和 `KazhdanLusztigCharacter` 的演示与现有测试主要集中在 **integral 的 full-affine** 工作流
- 一般情形下从 $\widehat\Delta_+^{\mathrm{re}}(\widehat\Lambda)$ 恢复 simple roots 与真正的 $\widehat W_\Lambda$ 生成元，还没有完全实现
- 当前 `character(...)` 返回的是 **grade-only** `FormalCharacter`，不是完整的按权 formal sum
- `Q_tilde` 的 affine 版本当前依赖 **bounded candidates + bounded stabilizer**，因此结果始终和给定的 `order` 有关
- 如果 `order` 太小，可能找不到 dominant conjugate，或者漏掉本应进入求和的代表元

如果你要继续扩展这个方向，最自然的下一步是：

1. 从 $\widehat\Delta_+^{\mathrm{re}}(\widehat\Lambda)$ 真正恢复一般 `W_\Lambda`
2. 细化 `W_\Lambda^0` 与 coset 最大代表元构造
3. 将当前 grade-only character 推广到更丰富的 character 表示
